# Body Segmentation

Lab Assignment from [AI for Beginners Curriculum](https://github.com/microsoft/ai-for-beginners).

In video production, for example, in weather forecasts, we often need to cut out a human image from camera and place it on top of some other footage. This is typically done using **chroma key** techniques, when a human is filmed in front of a uniform color background, which is then removed. In this lab, we will train a neural network model to cut out the human silhouette.

We will be using [Segmentation Full Body MADS Dataset](https://www.kaggle.com/datasets/tapakah68/segmentation-full-body-mads-dataset) from Kaggle. Download the dataset manually from Kaggle and unzip in into current directory.

In [ ]:
# Dataset path (nested structure after unzip)
dataset_path = 'segmentation_full_body_mads_dataset_1192_img/segmentation_full_body_mads_dataset_1192_img'

import os
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import transforms
from torch import nn
from torch import optim
from tqdm import tqdm
import numpy as np
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Explore the Dataset

Let's see how images in the dataset look like:

In [ ]:
img_path = os.path.join(dataset_path, 'images')
mask_path = os.path.join(dataset_path, 'masks')

# Check if dataset exists
if os.path.exists(img_path) and os.path.exists(mask_path):
    fnames = os.listdir(img_path)
    print(f"Found {len(fnames)} images in dataset")
    print(f"Images path: {img_path}")
    print(f"Masks path: {mask_path}")
else:
    print("Dataset not found!")
    print(f"Looking for: {dataset_path}")
    print("Please download from Kaggle:")
    print("https://www.kaggle.com/datasets/tapakah68/segmentation-full-body-mads-dataset")
    fnames = []

def load_image(img_name):
    img = plt.imread(os.path.join(img_path, img_name))
    mask = plt.imread(os.path.join(mask_path, img_name))
    return img, mask

In [ ]:
# Visualize sample images
if len(fnames) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, idx in enumerate([0, 5, 10, 20]):
        img, mask = load_image(fnames[idx])
        axes[0, i].imshow(img)
        axes[0, i].set_title(f'Image {idx}')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(mask)
        axes[1, i].set_title(f'Mask {idx}')
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()

## 2. Create Custom Dataset Class

In [ ]:
class BodySegmentationDataset(Dataset):
    def __init__(self, root, transform=None, img_size=(256, 256)):
        self.root = root
        self.transform = transform
        self.img_size = img_size
        
        self.img_dir = os.path.join(root, 'images')
        self.mask_dir = os.path.join(root, 'masks')
        
        self.images = sorted(os.listdir(self.img_dir))
        self.masks = sorted(os.listdir(self.mask_dir))
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])
        
        # Load image and mask
        img = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')  # Grayscale mask
        
        # Resize
        img = img.resize(self.img_size, Image.BILINEAR)
        mask = mask.resize(self.img_size, Image.NEAREST)
        
        # Convert to tensor
        img = transforms.ToTensor()(img)
        mask = transforms.ToTensor()(mask)
        
        # Binarize mask (threshold at 0.5)
        mask = (mask > 0.5).float()
        
        return img, mask

print("Dataset class defined.")

In [ ]:
# Create dataset and split
if os.path.exists(dataset_path):
    full_dataset = BodySegmentationDataset(dataset_path)
    
    # Split into train/test
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, test_size]
    )
    
    # Create dataloaders
    batch_size = 16
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
else:
    print("Please download the dataset first!")

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        
        # Encoder (downsampling path)
        self.enc_conv0 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.enc_bn0 = nn.BatchNorm2d(64)
        self.enc_conv1 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.enc_bn1 = nn.BatchNorm2d(64)
        self.pool0 = nn.MaxPool2d(2)
        
        self.enc_conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.enc_bn2 = nn.BatchNorm2d(128)
        self.enc_conv3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.enc_bn3 = nn.BatchNorm2d(128)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc_conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.enc_bn4 = nn.BatchNorm2d(256)
        self.enc_conv5 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.enc_bn5 = nn.BatchNorm2d(256)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc_conv6 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.enc_bn6 = nn.BatchNorm2d(512)
        self.enc_conv7 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.enc_bn7 = nn.BatchNorm2d(512)
        self.pool3 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck_conv0 = nn.Conv2d(512, 1024, kernel_size=3, padding=1)
        self.bottleneck_bn0 = nn.BatchNorm2d(1024)
        self.bottleneck_conv1 = nn.Conv2d(1024, 1024, kernel_size=3, padding=1)
        self.bottleneck_bn1 = nn.BatchNorm2d(1024)
        
        # Decoder (upsampling path)
        self.up0 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec_conv0 = nn.Conv2d(1024, 512, kernel_size=3, padding=1)
        self.dec_bn0 = nn.BatchNorm2d(512)
        self.dec_conv1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.dec_bn1 = nn.BatchNorm2d(512)
        
        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec_conv2 = nn.Conv2d(512, 256, kernel_size=3, padding=1)
        self.dec_bn2 = nn.BatchNorm2d(256)
        self.dec_conv3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.dec_bn3 = nn.BatchNorm2d(256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec_conv4 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.dec_bn4 = nn.BatchNorm2d(128)
        self.dec_conv5 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.dec_bn5 = nn.BatchNorm2d(128)
        
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec_conv6 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.dec_bn6 = nn.BatchNorm2d(64)
        self.dec_conv7 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.dec_bn7 = nn.BatchNorm2d(64)
        
        # Final output
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)
        
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Encoder
        e0 = self.relu(self.enc_bn0(self.enc_conv0(x)))
        e0 = self.relu(self.enc_bn1(self.enc_conv1(e0)))
        p0 = self.pool0(e0)
        
        e1 = self.relu(self.enc_bn2(self.enc_conv2(p0)))
        e1 = self.relu(self.enc_bn3(self.enc_conv3(e1)))
        p1 = self.pool1(e1)
        
        e2 = self.relu(self.enc_bn4(self.enc_conv4(p1)))
        e2 = self.relu(self.enc_bn5(self.enc_conv5(e2)))
        p2 = self.pool2(e2)
        
        e3 = self.relu(self.enc_bn6(self.enc_conv6(p2)))
        e3 = self.relu(self.enc_bn7(self.enc_conv7(e3)))
        p3 = self.pool3(e3)
        
        # Bottleneck
        b = self.relu(self.bottleneck_bn0(self.bottleneck_conv0(p3)))
        b = self.relu(self.bottleneck_bn1(self.bottleneck_conv1(b)))
        
        # Decoder with skip connections
        d0 = self.up0(b)
        d0 = torch.cat([d0, e3], dim=1)
        d0 = self.relu(self.dec_bn0(self.dec_conv0(d0)))
        d0 = self.relu(self.dec_bn1(self.dec_conv1(d0)))
        
        d1 = self.up1(d0)
        d1 = torch.cat([d1, e2], dim=1)
        d1 = self.relu(self.dec_bn2(self.dec_conv2(d1)))
        d1 = self.relu(self.dec_bn3(self.dec_conv3(d1)))
        
        d2 = self.up2(d1)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.relu(self.dec_bn4(self.dec_conv4(d2)))
        d2 = self.relu(self.dec_bn5(self.dec_conv5(d2)))
        
        d3 = self.up3(d2)
        d3 = torch.cat([d3, e0], dim=1)
        d3 = self.relu(self.dec_bn6(self.dec_conv6(d3)))
        d3 = self.relu(self.dec_bn7(self.dec_conv7(d3)))
        
        # Final output
        out = self.final_conv(d3)
        return out

model = UNet(in_channels=3, out_channels=1).to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

In [ ]:
# Loss function and optimizer
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

def dice_coefficient(pred, target, epsilon=1e-6):
    """Calculate Dice coefficient for segmentation evaluation"""
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    intersection = (pred * target).sum()
    dice = (2. * intersection + epsilon) / (pred.sum() + target.sum() + epsilon)
    return dice.item()

def iou_score(pred, target, epsilon=1e-6):
    """Calculate IoU (Intersection over Union)"""
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    iou = (intersection + epsilon) / (union + epsilon)
    return iou.item()

print("Loss and metrics defined.")

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    total_dice = 0
    
    for images, masks in tqdm(loader, desc='Training'):
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_dice += dice_coefficient(outputs, masks)
    
    return total_loss / len(loader), total_dice / len(loader)

def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    total_dice = 0
    total_iou = 0
    
    with torch.no_grad():
        for images, masks in tqdm(loader, desc='Validation'):
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            
            total_loss += loss.item()
            total_dice += dice_coefficient(outputs, masks)
            total_iou += iou_score(outputs, masks)
    
    return total_loss / len(loader), total_dice / len(loader), total_iou / len(loader)

print("Training functions defined.")

In [ ]:
# Training
num_epochs = 20
train_losses, val_losses = [], []
train_dices, val_dices = [], []

print(f"Starting training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    train_loss, train_dice = train_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss, val_dice, val_iou = validate(model, test_loader, loss_fn, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_dices.append(train_dice)
    val_dices.append(val_dice)
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}, Dice: {train_dice:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Dice: {val_dice:.4f}, IoU: {val_iou:.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Dice coefficient curve
axes[1].plot(train_dices, label='Train Dice')
axes[1].plot(val_dices, label='Val Dice')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Coefficient')
axes[1].set_title('Training and Validation Dice Score')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
def visualize_predictions(model, dataset, num_samples=5):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    with torch.no_grad():
        for i in range(num_samples):
            img, mask = dataset[i]
            img_input = img.unsqueeze(0).to(device)
            
            pred = model(img_input)
            pred = torch.sigmoid(pred).cpu().squeeze(0)
            pred_mask = (pred > 0.5).float()
            
            # Original image
            axes[i, 0].imshow(img.permute(1, 2, 0))
            axes[i, 0].set_title('Original Image')
            axes[i, 0].axis('off')
            
            # Ground truth mask
            axes[i, 1].imshow(mask.squeeze(), cmap='gray')
            axes[i, 1].set_title('Ground Truth')
            axes[i, 1].axis('off')
            
            # Predicted mask
            axes[i, 2].imshow(pred_mask.squeeze(), cmap='gray')
            axes[i, 2].set_title('Predicted Mask')
            axes[i, 2].axis('off')
            
            # Overlay
            img_np = img.permute(1, 2, 0).numpy()
            overlay = img_np.copy()
            overlay[pred_mask.squeeze().numpy() > 0.5] = [1, 0, 0]  # Red for segmented area
            blended = 0.7 * img_np + 0.3 * overlay
            axes[i, 3].imshow(blended)
            axes[i, 3].set_title('Overlay')
            axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_predictions(model, test_dataset, num_samples=5)

In [ ]:
def evaluate_metrics(model, loader, device):
    model.eval()
    total_dice = 0
    total_iou = 0
    total_pixel_acc = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            preds = torch.sigmoid(outputs)
            preds = (preds > 0.5).float()
            
            # Dice
            intersection = (preds * masks).sum()
            dice = (2. * intersection) / (preds.sum() + masks.sum() + 1e-6)
            total_dice += dice.item()
            
            # IoU
            union = preds.sum() + masks.sum() - intersection
            iou = intersection / (union + 1e-6)
            total_iou += iou.item()
            
            # Pixel accuracy
            correct = (preds == masks).float().sum()
            total = masks.numel()
            total_pixel_acc += (correct / total).item()
    
    n = len(loader)
    return {
        'Dice': total_dice / n,
        'IoU': total_iou / n,
        'Pixel Accuracy': total_pixel_acc / n
    }

metrics = evaluate_metrics(model, test_loader, device)
print("=" * 40)
print("Final Test Metrics:")
print("=" * 40)
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

In [ ]:
# Save the trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'body_segmentation_unet.pth')

print("Model saved to body_segmentation_unet.pth")

## Summary

In this lab, we:
1. Explored the body segmentation dataset
2. Created a custom PyTorch Dataset class
3. Implemented a U-Net architecture with skip connections
4. Trained the model using BCE Loss
5. Evaluated using Dice coefficient and IoU metrics
6. Visualized segmentation results

### Key Takeaways
- **U-Net** excels at segmentation by preserving spatial information through skip connections
- **Dice coefficient** and **IoU** are standard metrics for segmentation evaluation
- Binary segmentation uses **BCEWithLogitsLoss** as the loss function

### Applications
- Medical imaging (tumor segmentation, organ detection)
- Autonomous driving (road/lane detection)
- Video production (background removal, virtual backgrounds)
- Satellite imagery analysis

## 9. Save the Model

## 8. Calculate Final Metrics

## 7. Visualize Predictions

## 6. Plot Training Results

## 5. Training Loop

For binary segmentation, we use **Binary Cross-Entropy with Logits Loss** (BCEWithLogitsLoss).

## 4. Define U-Net Model

U-Net is the most popular architecture for image segmentation. It uses skip connections between encoder and decoder to preserve spatial information.

## 3. Split Data and Create DataLoaders